# NR PRACH channel extraction on XCZU67DR
Upload a repeated 139-element short-preamble useful symbol at 15.36 MS/s. One active CC and one RACH channel perform frequency translation and 4× decimation. Verify the 37-sample relative arrival time with a known-root cyclic matched filter. The receiver checks an independent clock window and every sample timestamp. A separate capture contains noise alone.

In [ ]:
import os, sys
from pathlib import Path
examples = Path(os.environ.get("PRACH_EXAMPLES", "../packages/pynq-dfe/examples/prach")).resolve()
sys.path.insert(0, str(examples))
from board import run
from model import LIMITS, CASES, correlation, unpack, predicted
import numpy as np
import matplotlib.pyplot as plt
print("Predeclared limits", LIMITS)

In [ ]:
out = Path(os.environ.get("PRACH_OUTPUT", "."))
result = run(Path(os.environ["PRACH_BITFILE"]).resolve(), out)
print("Software/hardware versions", result["versions"], "events", result["events"])
for row in result["rows"]:
    print(f"{row['test']:38s}  measured {row['measured']}  predicted {row['predicted']}  {'PASS' if row['passed'] else 'FAIL'}")

In [ ]:
fig, axes = plt.subplots(2, 1, constrained_layout=True, figsize=(8, 6))
for ax, case in zip(axes, ("shifted", "noise")):
    y = unpack(np.load(out / (case + "-capture.npy")))
    p = predicted(np.load(out / (case + "-input.npy")))
    for samples, label in [(p, "Model"), (y, "Board")]:
        ax.plot(correlation(samples)[3], label=label)
    ax.set(xlabel="Cyclic lag (output samples)", ylabel="Normalized correlation", title=case)
    ax.legend()
fig.savefig(out / "correlation.png")
plt.show()
assert all(row["passed"] for row in result["rows"])